### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import sys
sys.path.append('./utils')
from svg_processor import SVGSanitizer, SVGProcessor,svg_constraints
from svg_evaluator_siglip import SVGMetricEvaluator


This code could modify your python environment or operating system.

Review this code at https://www.kaggle.com/code/metric/svg-constraints/versions/1
or in your download cache at /home/vino/.cache/kagglehub/notebooks/metric/svg-constraints/output/versions/1

It is strongly recommended that you run this code within a container
such as Docker to provide a secure, isolated execution environment.
See https://www.kaggle.com/docs/packages for more information.

Do you want to proceed? (y)es/[no]:  y


In [3]:
import mlflow
import os
os.environ['MLFLOW_TRACKING_URI'] = './mlruns'
import gc
import pandas as pd
from svg_processor import SVGSanitizer, SVGProcessor, svg_constraints
from svg_evaluator_siglip import SVGMetricEvaluator
from vllm import LLM, SamplingParams
import  re
from tqdm import tqdm
tqdm.pandas()

class Model:
    def __init__(self):
        # MLflow experiment tracking setup
        self.experiment_name = "svg_score_15"
        mlflow.set_experiment(self.experiment_name)

        self.model_path = "./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1"
        self.model = LLM(
            model=self.model_path,
            dtype="float16",
            max_model_len=1024,
            gpu_memory_utilization=0.85
        )

        # Default generation parameters (can be overridden later in run_experiment)
        self.temperature = 0.5
        self.top_k = 40
        self.top_p = 0.95
        self.max_tokens = 1024

        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

    def close_model(self):
        del self.model
        gc.collect()

    def log_params_and_metrics(self, model_name, temperature, top_k, top_p, max_tokens, sl_score):
        """ Log parameters and metrics to MLflow """
        mlflow.log_param("model", model_name)
        mlflow.log_param("temperature", temperature)
        mlflow.log_param("top_k", top_k)
        mlflow.log_param("top_p", top_p)
        mlflow.log_param("max_tokens", max_tokens)
        mlflow.log_metric("siglip_score", sl_score)
        
    def get_response(self, description, temperature, top_k, top_p, max_tokens):
        alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. 
        Write a response that appropriately completes the request.

                ### Instruction:
                Please write a SVG code for the given input.

                ### Input:
                {}

                ### Response:
                """

        formatted_input = alpaca_prompt.format(description)
        sampling_params = SamplingParams(temperature=temperature, top_k=top_k, top_p=top_p, max_tokens=max_tokens)
        outputs = self.model.generate([formatted_input], sampling_params)

        # suitable for batch inputs as well        
        for output in outputs:
            prompt = output.prompt
            generated_text = output.outputs[0].text
        return generated_text

    def predict(self, description: str, temperature, top_k, top_p, max_tokens) -> str:
        output_decoded = self.get_response(description, temperature, top_k, top_p, max_tokens)
        base_svg_code = SVGProcessor.clean_and_extract_svgs(output_decoded, self.default_svg)
        clean_svg_code = self.sanitizer.enforce_constraints(base_svg_code)
        return SVGProcessor.svg_conversion_check(description, clean_svg_code, self.default_svg)

    def run_experiment(self, df, model_name, temperature=None, top_k=None, top_p=None, max_tokens=None):
        """ Log experiment and track prediction & evaluation """
        # Use passed parameters, or default to the instance's values
        temperature = temperature or self.temperature
        top_k = top_k or self.top_k
        top_p = top_p or self.top_p
        max_tokens = max_tokens or self.max_tokens
        
        with mlflow.start_run():
            # Generate SVG code
            df['svg'] = df['description'].progress_apply(lambda x: self.predict(x, temperature, top_k, top_p, max_tokens))

            # Generate sl score
            df['sl_score'] = df.progress_apply(lambda row: SVGMetricEvaluator().svg_metric(row['description'], row['svg']), axis=1)

            sl_score = df['sl_score'].mean()
            
            # Log parameters and metrics
            self.log_params_and_metrics(model_name, temperature, top_k, top_p, max_tokens, sl_score)
            
            return df, sl_score



INFO 04-07 23:24:46 [__init__.py:239] Automatically detected platform cuda.


In [4]:
#model instance 
model = Model()

WARNING 04-07 23:24:47 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 04-07 23:24:51 [config.py:585] This model supports multiple tasks: {'reward', 'generate', 'classify', 'score', 'embed'}. Defaulting to 'generate'.
INFO 04-07 23:24:51 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-07 23:24:53 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1', speculative_config=None, tokenizer='./lora/lora_16bit_merged_3b_r128_s1000_i1000_v1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_b

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 04-07 23:24:55 [loader.py:447] Loading weights took 2.12 seconds
INFO 04-07 23:24:56 [gpu_model_runner.py:1186] Model loading took 6.0160 GB and 2.273539 seconds
INFO 04-07 23:25:01 [backends.py:415] Using cache directory: /home/vino/.cache/vllm/torch_compile_cache/84ad9dae08/rank_0_0 for vLLM's torch.compile
INFO 04-07 23:25:01 [backends.py:425] Dynamo bytecode transform time: 5.81 s
INFO 04-07 23:25:02 [backends.py:115] Directly load the compiled graph for shape None from the cache
INFO 04-07 23:25:06 [monitor.py:33] torch.compile takes 5.81 s in total
INFO 04-07 23:25:07 [kv_cache_utils.py:566] GPU KV cache size: 18,752 tokens
INFO 04-07 23:25:07 [kv_cache_utils.py:569] Maximum concurrency for 1,024 tokens per request: 18.31x
INFO 04-07 23:25:23 [gpu_model_runner.py:1534] Graph capturing finished in 16 secs, took 0.42 GiB
INFO 04-07 23:25:23 [core.py:151] init engine (profile, create kv cache, warmup model) took 27.84 seconds


In [5]:
#load df & score
df=pd.read_csv('./drawing-with-llms/train.csv',header=[0])

In [6]:
model_name = re.sub(r'[^a-zA-Z0-9]', '_', model.model_path)
df_, sl_score = model.run_experiment(df, model_name, temperature=0.5, top_k=50, top_p=0.95, max_tokens=1024)
print('sl_score',sl_score)

  0%|                                                    | 0/15 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:08<00:00,  8.97s/it, est. speed input: 6.80 t
 13%|█████▊                                      | 2/15 [00:09<00:58,  4.53s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.81s/it, est. speed input: 16.80 
 20%|████████▊                                   | 3/15 [00:12<00:50,  4.23s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.90s/it, est. speed input: 15.91 
 27%|███████████▋                                | 4/15 [00:16<00:45,  4.11s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:07<00:00,  7.23s/it, est. speed input: 9.41 t
 33%|██████████████▋                    

Using device: cuda


 13%|█████▊                                      | 2/15 [00:05<00:33,  2.58s/it]

Using device: cuda


 20%|████████▊                                   | 3/15 [00:10<00:42,  3.56s/it]

...

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:09<00:00,  4.99s/it]

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:15<00:00,  5.01s/it]

sl_score 0.3499771252203528


In [7]:
df_, sl_score = model.run_experiment(df, model_name, temperature=0.4, top_k=30, top_p=0.95, max_tokens=1024)
print('sl_score',sl_score)

  0%|                                                    | 0/15 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:06<00:00,  6.73s/it, est. speed input: 9.06 t
 13%|█████▊                                      | 2/15 [00:06<00:43,  3.37s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:07<00:00,  7.02s/it, est. speed input: 9.12 t
 20%|████████▊                                   | 3/15 [00:13<00:58,  4.89s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.50s/it, est. speed input: 17.71 
 27%|███████████▋                                | 4/15 [00:17<00:48,  4.38s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.84s/it, est. speed input: 11.64 
 33%|██████████████▋                    

Using device: cuda


 13%|█████▊                                      | 2/15 [00:04<00:30,  2.35s/it]

Using device: cuda


 20%|████████▊                                   | 3/15 [00:09<00:41,  3.42s/it]

...

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:10<00:00,  4.94s/it]

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:15<00:00,  5.05s/it]

sl_score 0.3679591852451116


In [8]:
df_, sl_score = model.run_experiment(df, model_name, temperature=0.5, top_k=20, top_p=0.95, max_tokens=1024)
print('sl_score',sl_score)

  0%|                                                    | 0/15 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:07<00:00,  7.82s/it, est. speed input: 7.80 t
 13%|█████▊                                      | 2/15 [00:07<00:50,  3.92s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:06<00:00,  6.18s/it, est. speed input: 10.36 
 20%|████████▊                                   | 3/15 [00:14<00:58,  4.86s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.46s/it, est. speed input: 25.26 
 27%|███████████▋                                | 4/15 [00:16<00:43,  3.97s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.79s/it, est. speed input: 11.75 
 33%|██████████████▋                    

Using device: cuda


 13%|█████▊                                      | 2/15 [00:05<00:35,  2.70s/it]

Using device: cuda


 20%|████████▊                                   | 3/15 [00:10<00:42,  3.54s/it]

...

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:09<00:00,  5.01s/it]

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:14<00:00,  4.96s/it]

sl_score 0.14740819625719884


In [9]:
df_, sl_score = model.run_experiment(df, model_name, temperature=0.5, top_k=30, top_p=0.95, max_tokens=1024)
print('sl_score',sl_score)

  0%|                                                    | 0/15 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:10<00:00, 10.27s/it, est. speed input: 5.94 t
 13%|█████▊                                      | 2/15 [00:10<01:06,  5.14s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:07<00:00,  7.86s/it, est. speed input: 8.15 t
 20%|████████▊                                   | 3/15 [00:18<01:15,  6.27s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.75s/it, est. speed input: 16.54 
 27%|███████████▋                                | 4/15 [00:21<00:58,  5.33s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:10<00:00, 10.91s/it, est. speed input: 6.24 t
 33%|██████████████▋                    

Using device: cuda


 13%|█████▊                                      | 2/15 [00:05<00:32,  2.50s/it]

Using device: cuda


 20%|████████▊                                   | 3/15 [00:10<00:45,  3.77s/it]

...

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:13<00:00,  5.42s/it]

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:18<00:00,  5.24s/it]

sl_score 0.3309140837493384


In [10]:
df_, sl_score = model.run_experiment(df, model_name, temperature=0.5, top_k=40, top_p=0.95, max_tokens=1024)
print('sl_score',sl_score)

  0%|                                                    | 0/15 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:08<00:00,  8.35s/it, est. speed input: 7.30 t
 13%|█████▊                                      | 2/15 [00:08<00:54,  4.18s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:07<00:00,  7.87s/it, est. speed input: 8.14 t
 20%|████████▊                                   | 3/15 [00:16<01:08,  5.72s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.89s/it, est. speed input: 10.53 
 27%|███████████▋                                | 4/15 [00:22<01:03,  5.79s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:07<00:00,  7.63s/it, est. speed input: 8.91 t
 33%|██████████████▋                    

Failed to convert orange corduroy overalls due to , Returning default SVG.



cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:10<00:00, 10.46s/it, est. speed input: 6.12 t
 47%|████████████████████▌                       | 7/15 [00:47<01:02,  7.82s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.38s/it, est. speed input: 14.63 
 53%|███████████████████████▍                    | 8/15 [00:51<00:47,  6.74s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:06<00:00,  6.42s/it, est. speed input: 9.65 t
 60%|██████████████████████████▍                 | 9/15 [00:57<00:39,  6.64s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:11<00:00, 11.39s/it, est. speed input: 5.71 t
ERROR:root:SVG Parse Error: Unescaped '<' not allowed in attributes values, line 25, column 644 (<string>, line 25). Ret

Using device: cuda


 13%|█████▊                                      | 2/15 [00:05<00:32,  2.51s/it]

...

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:13<00:00,  5.20s/it]

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:17<00:00,  5.19s/it]

sl_score 0.3020005675161548


In [11]:
df_, sl_score = model.run_experiment(df, model_name, temperature=0.5, top_k=50, top_p=0.95, max_tokens=1024)
print('sl_score',sl_score)

  0%|                                                    | 0/15 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:09<00:00,  9.30s/it, est. speed input: 6.56 t
 13%|█████▊                                      | 2/15 [00:09<01:00,  4.66s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.21s/it, est. speed input: 12.28 
 20%|████████▊                                   | 3/15 [00:14<00:58,  4.89s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.14s/it, est. speed input: 19.75 
 27%|███████████▋                                | 4/15 [00:17<00:46,  4.24s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:07<00:00,  7.35s/it, est. speed input: 9.25 t
 33%|██████████████▋                    

Using device: cuda


 13%|█████▊                                      | 2/15 [00:05<00:33,  2.57s/it]

Using device: cuda


 20%|████████▊                                   | 3/15 [00:10<00:43,  3.62s/it]

...

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:11<00:00,  5.15s/it]

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:17<00:00,  5.13s/it]

sl_score 0.3453117136164848


In [12]:
df_, sl_score = model.run_experiment(df, model_name, temperature=0.5, top_k=40, top_p=0.9, max_tokens=1024)
print('sl_score',sl_score)

  0%|                                                    | 0/15 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:07<00:00,  7.39s/it, est. speed input: 8.25 t
 13%|█████▊                                      | 2/15 [00:07<00:48,  3.70s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.79s/it, est. speed input: 11.06 
 20%|████████▊                                   | 3/15 [00:13<00:54,  4.57s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.20s/it, est. speed input: 19.40 
 27%|███████████▋                                | 4/15 [00:16<00:44,  4.06s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:09<00:00,  9.87s/it, est. speed input: 6.89 t
 33%|██████████████▋                    

Using device: cuda


 13%|█████▊                                      | 2/15 [00:05<00:36,  2.79s/it]

Using device: cuda


 20%|████████▊                                   | 3/15 [00:10<00:44,  3.69s/it]

...

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:10<00:00,  5.00s/it]

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:15<00:00,  5.04s/it]

sl_score 0.37060701048723127


In [13]:
df_, sl_score = model.run_experiment(df, model_name, temperature=0.6, top_k=30, top_p=0.95, max_tokens=1024)
print('sl_score',sl_score)

  0%|                                                    | 0/15 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.54s/it, est. speed input: 11.02 
 13%|█████▊                                      | 2/15 [00:05<00:36,  2.77s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.45s/it, est. speed input: 11.75 
 20%|████████▊                                   | 3/15 [00:10<00:46,  3.89s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.17s/it, est. speed input: 19.56 
 27%|███████████▋                                | 4/15 [00:14<00:39,  3.62s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:06<00:00,  6.11s/it, est. speed input: 11.14 
 33%|██████████████▋                    

Using device: cuda


 13%|█████▊                                      | 2/15 [00:04<00:30,  2.37s/it]

Using device: cuda


 20%|████████▊                                   | 3/15 [00:09<00:42,  3.52s/it]

...

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:11<00:00,  5.03s/it]

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:16<00:00,  5.10s/it]

sl_score 0.33078926447844886


In [14]:
df_, sl_score = model.run_experiment(df, model_name, temperature=0.5, top_k=30, top_p=0.95, max_tokens=2048)
print('sl_score',sl_score)

  0%|                                                    | 0/15 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:10<00:00, 10.54s/it, est. speed input: 5.79 t
 13%|█████▊                                      | 2/15 [00:10<01:08,  5.27s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.35s/it, est. speed input: 14.70 
 20%|████████▊                                   | 3/15 [00:14<00:58,  4.89s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.35s/it, est. speed input: 18.52 
 27%|███████████▋                                | 4/15 [00:18<00:47,  4.32s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:07<00:00,  7.33s/it, est. speed input: 9.28 t
 33%|██████████████▋                    

Using device: cuda


 13%|█████▊                                      | 2/15 [00:05<00:38,  3.00s/it]

Using device: cuda


 20%|████████▊                                   | 3/15 [00:10<00:46,  3.83s/it]

...

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:14<00:00,  5.09s/it]

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:19<00:00,  5.29s/it]

sl_score 0.3392732402414595


In [15]:
#model.close_model()

In [16]:
df_, sl_score = model.run_experiment(df, model_name, temperature=0.5, top_k=60, top_p=0.95, max_tokens=2048)
print('sl_score',sl_score)

  0%|                                                    | 0/15 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:06<00:00,  6.95s/it, est. speed input: 8.78 t
 13%|█████▊                                      | 2/15 [00:06<00:45,  3.48s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.43s/it, est. speed input: 11.78 
 20%|████████▊                                   | 3/15 [00:12<00:51,  4.30s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.04s/it, est. speed input: 15.36 
 27%|███████████▋                                | 4/15 [00:16<00:46,  4.20s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:06<00:00,  6.13s/it, est. speed input: 11.09 
 33%|██████████████▋                    

Using device: cuda


 13%|█████▊                                      | 2/15 [00:05<00:36,  2.78s/it]

Using device: cuda


 20%|████████▊                                   | 3/15 [00:10<00:45,  3.76s/it]

...

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:10<00:00,  4.94s/it]

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:15<00:00,  5.05s/it]

sl_score 0.3758944315625097


In [17]:
df_, sl_score = model.run_experiment(df, model_name, temperature=0.5, top_k=60, top_p=0.95, max_tokens=1024)
print('sl_score',sl_score)

  0%|                                                    | 0/15 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:07<00:00,  7.58s/it, est. speed input: 8.05 t
 13%|█████▊                                      | 2/15 [00:07<00:49,  3.79s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:04<00:00,  4.47s/it, est. speed input: 14.32 
 20%|████████▊                                   | 3/15 [00:12<00:48,  4.08s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:02<00:00,  2.05s/it, est. speed input: 30.28 
 27%|███████████▋                                | 4/15 [00:14<00:36,  3.32s/it]
cessed prompts:   0%| | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
Processed prompts: 100%|█| 1/1 [00:05<00:00,  5.60s/it, est. speed input: 12.13 
 33%|██████████████▋                    

Using device: cuda


 13%|█████▊                                      | 2/15 [00:04<00:32,  2.50s/it]

Using device: cuda


 20%|████████▊                                   | 3/15 [00:11<00:49,  4.09s/it]

...

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:11<00:00,  4.93s/it]

Using device: cuda


100%|███████████████████████████████████████████| 15/15 [01:16<00:00,  5.07s/it]

sl_score 0.27962686547663246
